# Скоринг, SHAP и sales-аргументы

Пайплайн после обучения: топ-3 продукта → SHAP-признаки → Jinja-промпт → аргумент (mock или LLM).

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from scoring import PropensityScorer, load_onboarding_clients
from explain import explain_client_product, contributions_to_dict
from llm_sales_pipeline import render_sales_prompt, generate_sales_argument

clients = load_onboarding_clients()
scorer = PropensityScorer()
client = clients.iloc[0]
client_id = 0

In [ ]:
top3 = scorer.score_client(client, top_k=3)
pd.DataFrame([t.__dict__ for t in top3])

In [ ]:
pid = top3[0].product_id
feats = explain_client_product(
    scorer.pipe, client, pid,
    cat_features=scorer.cat_features,
    num_features=scorer.num_features,
)
pd.DataFrame(contributions_to_dict(feats))

In [ ]:
prompt = render_sales_prompt(
    client_id=client_id,
    client=client,
    product_id=pid,
    product_name=top3[0].product_name,
    propensity_score=top3[0].propensity_score,
    top_features=feats,
)
print(prompt[:1200], "...")

In [ ]:
arg = generate_sales_argument(
    client_id=client_id,
    client=client,
    product_id=pid,
    product_name=top3[0].product_name,
    propensity_score=top3[0].propensity_score,
    top_features=feats,
    use_mock=True,
)
print(json.dumps(arg, ensure_ascii=False, indent=2))

In [ ]:
# Загрузка батч-результатов
sales_path = ROOT / "data" / "sales_arguments.jsonl"
if sales_path.exists():
    rows = [json.loads(l) for l in sales_path.open(encoding="utf-8")]
    sample = rows[0]["sales_content"]
    print("Пример аргумента:", sample["sales_argument_ru"][:300])